# SCREEN: компактные тканевые метаданные для baseline

E2E: homologous ELS pairs → SCREEN → потоковая агрегация → Parquet dataset.
В baseline остаются только `ccre_id`, `tissue`, `assay`, `zscore`. Повторные измерения одной ткани/assay для одного cCRE схлопываются по `max`.

Важно: сырой SCREEN JSON больше не накапливается в RAM и новые огромные `.json`-батчи не сохраняются. Уже скачанный `.screen_cache` переиспользуется по одному батчу.

In [9]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "eclare").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from eclare.data.screen import fetch_human_mouse_tissue_metadata

DATA_DIR = project_root / "data/encode"
OUTPUT_DIR = project_root / "data/processed/screen_tissue"

# Быстрый baseline: берём первые 20k human cCRE по тому же sorted-ID порядку,
# поэтому почти весь уже скачанный human .screen_cache переиспользуется.
# Это warm-start subset, а не unbiased evaluation sample. Для полного датасета: None.
MAX_HUMAN_IDS = 20_000

# None = сохраняем все assays, но уже без biosample/sample_type/life_stage.
# Позже можно указать конкретные assay строками, не меняя код загрузчика.
ASSAYS = None
BATCH_SIZE = 200

In [10]:
import os
from getpass import getpass
import dotenv

dotenv.load_dotenv(project_root / ".env")

# Ключ остаётся в окружении kernel, а не в сохранённом notebook.
if not os.environ.get("SCREEN_API_KEY"):
    os.environ["SCREEN_API_KEY"] = getpass("SCREEN API key: ")

## Загрузка

Каждый SCREEN batch сразу превращается в компактную таблицу `ccre_id × tissue × assay` и записывается отдельным Parquet shard. Поэтому объём RAM ограничен одним батчем, а повторный запуск продолжает с готовых shard-файлов.

`max` по повторным biosample/experiment/file — осознанное упрощение для первого baseline: оно отвечает на вопрос, наблюдалась ли сильная активность cCRE хотя бы в одном измерении данной ткани и assay.

In [11]:
pairs, human_dataset, mouse_dataset = fetch_human_mouse_tissue_metadata(
    DATA_DIR / "hg38-mm10-Homologous.tsv",
    DATA_DIR / "GRCh38-cCREs.ELS.bed",
    DATA_DIR / "mm10-cCREs.ELS.bed",
    output_dir=OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    assays=ASSAYS,
    max_human_ids=MAX_HUMAN_IDS,
    raw_cache_dir=project_root / ".screen_cache",
)

SCREEN tissue mm10: 100%|██████████| 21003/21003 [07:11<00:00, 48.65cCRE/s]


## Проверка результата

Не загружаем весь dataset в pandas. `pyarrow.dataset` быстро считает строки по Parquet metadata и читает только маленький preview.

In [12]:
import pyarrow.dataset as ds

human_ds = ds.dataset(human_dataset, format="parquet", exclude_invalid_files=True)
mouse_ds = ds.dataset(mouse_dataset, format="parquet", exclude_invalid_files=True)

print(f"ELS pairs in baseline subset: {len(pairs):,}")
print(f"Unique human cCREs: {pairs.human_id.nunique():,}")
print(f"Unique mouse cCREs: {pairs.mouse_id.nunique():,}")
print(f"Human compact rows: {human_ds.count_rows():,}")
print(f"Mouse compact rows: {mouse_ds.count_rows():,}")
print("Human dataset:", human_dataset)
print("Mouse dataset:", mouse_dataset)

display(human_ds.head(10).to_pandas())
display(mouse_ds.head(10).to_pandas())


ELS pairs in baseline subset: 22,842
Unique human cCREs: 20,000
Unique mouse cCREs: 21,003
Human compact rows: 63,980,000
Mouse compact rows: 12,412,773
Human dataset: c:\Projects\eclare\data\processed\screen_tissue\grch38-21564c80a8c9ebaf
Mouse dataset: c:\Projects\eclare\data\processed\screen_tissue\mm10-647e6942a349e062


,ccre_id,tissue,assay,zscore
0,EH38E0048077,breast,DNase,0.134055
1,EH38E0048077,large intestine,DNase,-1.035755
2,EH38E0048077,brain,DNase,0.760830
3,EH38E0048077,brain,DNase,0.437904
4,EH38E0048077,brain,DNase,0.526583
5,EH38E0048077,brain,DNase,0.812102
6,EH38E0048077,brain,DNase,0.825565
7,EH38E0048077,brain,DNase,0.342288
8,EH38E0048077,blood,DNase,0.482263
9,EH38E0048077,brain,DNase,0.571847


,ccre_id,tissue,assay,zscore
0,EM10E0107621,liver,DNase,-1.075585
1,EM10E0107621,brain,DNase,-2.030843
2,EM10E0107621,lung,DNase,-0.071748
3,EM10E0107621,brain,DNase,-1.770257
4,EM10E0107621,urinary bladder,DNase,-0.484136
5,EM10E0107621,brain,DNase,-1.521034
6,EM10E0107621,brain,DNase,-1.612713
7,EM10E0107621,lung,DNase,-1.193587
8,EM10E0107621,muscle,DNase,-0.846956
9,EM10E0107621,spleen,DNase,-2.363030


## Сохранение пар

Тканевые метаданные уже сохранены потоково в каталогах выше. Отдельно сохраняем только соответствующие homologous pairs.

In [13]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pairs_path = OUTPUT_DIR / "orthologous_els_baseline.parquet"
pairs.to_parquet(pairs_path, index=False)
print("Pairs:", pairs_path)

Pairs: c:\Projects\eclare\data\processed\screen_tissue\orthologous_els_baseline.parquet


## Когда baseline заработает

Для полного прогона поменяйте только `MAX_HUMAN_IDS = None` и перезапустите ячейку загрузки. Уже созданный baseline dataset останется отдельным, а старый raw `.screen_cache` не будет целиком загружаться в память.

In [14]:
!pip install duckdb

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
    --------------------------------------- 0.3/13.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/13.2 MB 1.7 MB/s eta 0:00:08
   ---- ----------------------------------- 1.6/13.2 MB 3.1 MB/s eta 0:00:04
   ---------- ----------------------------- 3.4/13.2 MB 5.0 MB/s eta 0:00:02
   --------------- ------------------------ 5.2/13.2 MB 6.1 MB/s eta 0:00:02
   --------------------- ------------------ 7.1/13.2 MB 6.3 MB/s eta 0:00:01
   --------------------------- ------------ 9.2/13.2 MB 7.0 MB/s eta 0:00:01
   ---------------------------------- ----- 11.5/13.2 MB 7.6 MB/s eta 0:00:01
   ---------------------------------------- 13.2/13.2 MB 7.8 MB/s  0:00:02
